# Notebook 10 — Feature Importance
Feature Selection (Notebook 9) decides *what to keep*. Feature Importance explains
*how much* each retained feature contributes, and — critically — **why that's not the
same as causation**.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

customers = pd.read_csv("./telecom_customers.csv", parse_dates=["signup_date"])
ref = pd.Timestamp("2024-06-30")
customers["tenure_days"] = (ref - customers["signup_date"]).dt.days
customers["total_charges"] = customers["total_charges"].fillna(customers["monthly_charges"])
customers["charge_per_tenure_month"] = customers["monthly_charges"] / customers["tenure_months"].replace(0,1)
customers["contract_ordinal"] = customers["contract"].map({"Month-to-month":0,"One year":1,"Two year":2})
customers["is_electronic_check"] = (customers["payment_method"]=="Electronic check").astype(int)
customers["is_fiber"] = (customers["internet_service"]=="Fiber optic").astype(int)
customers["churn_binary"] = (customers["churn"]=="Yes").astype(int)

feature_cols = ["monthly_charges","tenure_months","total_charges","charge_per_tenure_month",
                 "contract_ordinal","is_electronic_check","is_fiber","senior_citizen"]
X = customers[feature_cols].fillna(0)
y = customers["churn_binary"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
X_train.shape, X_test.shape

## 1. Decision Tree Feature Importance

A single Decision Tree computes importance as the total reduction in impurity (e.g.
Gini) that each feature is responsible for, summed across every split where it's used.

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
dt_importance = pd.Series(dt.feature_importances_, index=feature_cols).sort_values(ascending=False)
dt_importance.plot(kind="barh", figsize=(6,4), color="#4C72B0", title="Decision Tree Feature Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 2. Random Forest Feature Importance

Averages impurity-reduction importance across many trees — much more stable than a
single tree, since it's less sensitive to the specific splits any one tree happened
to make.

In [ ]:
rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
rf_importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
rf_importance.plot(kind="barh", figsize=(6,4), color="#DD8452", title="Random Forest Feature Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Permutation Importance

Measures importance by **shuffling one feature's values at a time** on held-out data
and observing how much model performance drops. This is model-agnostic and, unlike
impurity-based importance, isn't biased toward high-cardinality numeric features.

In [ ]:
perm = permutation_importance(rf, X_test, y_test, n_repeats=20, random_state=42, scoring="roc_auc")
perm_importance = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)
perm_importance.plot(kind="barh", figsize=(6,4), color="#55A868", title="Permutation Importance (ROC-AUC drop)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Feature Importance vs Feature Selection

These are related but distinct steps:
- **Feature Selection** (Notebook 9) is a *decision process* — keep or discard.
- **Feature Importance** is a *diagnostic/explanatory tool* — given a set of already-
  selected features, rank how much each one drives predictions.

You can use importance scores *as input to* a selection decision (e.g. drop anything
below a threshold), but importance alone doesn't tell you about redundancy between
correlated features the way a correlation-based selection method does.

## 5. Interpreting Feature Importance Correctly

In [ ]:
comparison = pd.DataFrame({
    "decision_tree": dt_importance,
    "random_forest": rf_importance,
    "permutation": perm_importance
}).sort_values("random_forest", ascending=False)
comparison

## 6. Why High Importance ≠ Causation

This is one of the most important conceptual points in the whole sprint, and one you
should always be able to explain in a review:

`contract_ordinal` and `is_electronic_check` consistently rank as high-importance —
but that does **not** mean switching a customer's payment method to autopay will
*cause* them to stop churning. Feature importance reflects **statistical
association captured by the model**, not a causal mechanism.

**Why this distinction matters in practice:**
- These features are likely **proxies** for an underlying latent factor — e.g.
  customer engagement/financial stability — that drives both the payment method
  choice *and* the churn decision independently.
- Confounding variables (like disposable income, which we don't have in this dataset)
  can inflate the apparent importance of a feature that merely correlates with the
  true cause.
- Acting on importance as if it were causation (e.g. "force everyone onto autopay to
  reduce churn") without a controlled experiment (A/B test) can waste budget or even
  backfire.

**Senior engineer takeaway:** feature importance tells you *where the model is
looking*, which is invaluable for debugging and building trust in the model — but
translating that into a causal business action requires either domain knowledge, a
causal inference method, or a genuine randomized experiment.

## Limitations of Feature Importance

- Impurity-based importance (Decision Tree/Random Forest) is biased toward numeric /
  high-cardinality features
- Correlated features "split" importance between themselves, making each look less
  important individually than the underlying signal actually is
- Permutation importance can be misleading if features are highly correlated
  (shuffling one still leaves the model able to use its correlated partner)
- None of these methods establish causality — only association within this model